In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer



root = Path().resolve().parent
file_path = root / "_docs" / "dummytext.txt"

load_dotenv()
SERVER_AI_URL = os.getenv("SERVER_AI_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

loader = TextLoader(file_path=file_path)
docs = loader.load()
print(f"documento cargado.")


text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=24)
documents = text_splitter.split_documents(documents=docs)
print(f"numero de chunks: {len(documents)}")

# Fail
# model = "gpt-oss:20b"
# llm = ChatOllama(model=model, temperature=0.2, base_url=SERVER_AI_URL)

# model = "gpt-4o-2024-08-06"
# llm = ChatOpenAI(temperature=0.1, api_key=OPENAI_API_KEY)

model = "deepseek-r1:14b"
llm = ChatOllama(model=model, temperature=0.2, base_url=SERVER_AI_URL)

llm_transformer = LLMGraphTransformer(llm=llm)
graph_documents = llm_transformer.convert_to_graph_documents(documents)


for graph_document in graph_documents:
    print(f"nodes: \n {graph_document.nodes}")
    print(f"relationships: \n {graph_document.relationships}")
    print("\n"*3)





documento cargado.
numero de chunks: 73
nodes: 
 [Node(id="Amico'S_Family", type='Family', properties={}), Node(id='Love', type='Concept', properties={}), Node(id='Tradition', type='Concept', properties={})]
relationships: 
 [Relationship(source=Node(id="Amico'S_Family", type='Family', properties={}), target=Node(id='Love', type='Concept', properties={}), type='ASSOCIATED_WITH', properties={}), Relationship(source=Node(id="Amico'S_Family", type='Family', properties={}), target=Node(id='Tradition', type='Concept', properties={}), type='ASSOCIATED_WITH', properties={})]




nodes: 
 [Node(id='Santa_Caterina', type='Village', properties={}), Node(id='Sicily', type='Island', properties={}), Node(id='Caruso_Family', type='Family', properties={})]
relationships: 
 [Relationship(source=Node(id='Santa_Caterina', type='Village', properties={}), target=Node(id='Sicily', type='Island', properties={}), type='LOCATED_IN', properties={}), Relationship(source=Node(id='Caruso_Family', type='Family', p

In [2]:
from langchain_neo4j.graphs.neo4j_graph  import Neo4jGraph
# url="neo4j://localhost:7687" "http://localhost:7687/"  
graph = Neo4jGraph(
    username="neo4j", 
    password="langchain", 
    url="bolt://localhost:7687"     #"bolt://localhost:7687"  # Cambia http:// por bolt://
)
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)


In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_neo4j.vectorstores.neo4j_vector import Neo4jVector


model_emb = "bge-m3:latest"
embeddings = OllamaEmbeddings(model=model_emb, base_url=SERVER_AI_URL)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    url="bolt://localhost:7687",
    username="neo4j",
    password="langchain"
)
vector_retriever = vector_index.as_retriever()


In [7]:
from neo4j import GraphDatabase


driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id` 
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session() as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    pass

# Close the driver connection
driver.close()

KeyError: 'NEO4J_URI'